In [1]:
import pandas as pd
pd.set_option('display.max_columns', 30)

### Load Dataset

In [6]:
path = '/content/balance_selected_100_companies .csv'

In [7]:
df = pd.read_csv(path)

In [6]:
df.shape

(500, 13)

In [8]:
df.head()

,Ticker,SimFinId,Fiscal Year,"Cash, Cash Equivalents & Short Term Investments",Total Current Assets,Total Assets,Total Current Liabilities,Total Liabilities,Total Equity,balance_diff,balance_error_pct,Short Term Debt,Long Term Debt
0,ABBV,61199,2020,8.479000e+09,2.417300e+10,150565000000,2.866100e+10,1.374680e+11,13097000000,0.0,0.0,8.502000e+09,7.755400e+10
1,ABBV,61199,2021,9.830000e+09,2.792800e+10,146529000000,3.519400e+10,1.310930e+11,15436000000,0.0,0.0,1.249500e+10,6.418900e+10
2,ABBV,61199,2022,9.229000e+09,2.846300e+10,138805000000,2.953800e+10,1.215180e+11,17287000000,0.0,0.0,4.136000e+09,5.913500e+10
3,ABBV,61199,2023,1.281600e+10,3.300200e+10,134711000000,3.784100e+10,1.243140e+11,10397000000,0.0,0.0,7.191000e+09,5.219400e+10
4,ABBV,61199,2024,5.555000e+09,2.558200e+10,135161000000,3.874900e+10,1.317970e+11,3364000000,0.0,0.0,6.804000e+09,6.034000e+10


### Copy dataset to create new feature

In [8]:
# Create a copy of the original DataFrame for feature engineering
df_features = df.copy()

# Sort the DataFrame by Ticker and Fiscal Year, which is crucial for calculating growth metrics
df_features = df_features.sort_values(by=['Ticker', 'Fiscal Year']).reset_index(drop=True)

print("DataFrame prepared for feature engineering.")
# Display the first few rows of the sorted DataFrame to verify
display(df_features.head())

DataFrame prepared for feature engineering.


,Ticker,SimFinId,Fiscal Year,"Cash, Cash Equivalents & Short Term Investments",Total Current Assets,Total Assets,Total Current Liabilities,Total Liabilities,Total Equity,balance_diff,balance_error_pct,Short Term Debt,Long Term Debt
0,ABBV,61199,2020,8.479000e+09,2.417300e+10,150565000000,2.866100e+10,1.374680e+11,13097000000,0.0,0.0,8.502000e+09,7.755400e+10
1,ABBV,61199,2021,9.830000e+09,2.792800e+10,146529000000,3.519400e+10,1.310930e+11,15436000000,0.0,0.0,1.249500e+10,6.418900e+10
2,ABBV,61199,2022,9.229000e+09,2.846300e+10,138805000000,2.953800e+10,1.215180e+11,17287000000,0.0,0.0,4.136000e+09,5.913500e+10
3,ABBV,61199,2023,1.281600e+10,3.300200e+10,134711000000,3.784100e+10,1.243140e+11,10397000000,0.0,0.0,7.191000e+09,5.219400e+10
4,ABBV,61199,2024,5.555000e+09,2.558200e+10,135161000000,3.874900e+10,1.317970e+11,3364000000,0.0,0.0,6.804000e+09,6.034000e+10


### 1. Current Ratio

**Meaning:** Current Assets / Current Liabilities

**Documentation:** This ratio measures a company's ability to pay off its short-term liabilities with its short-term assets. A higher ratio indicates better short-term liquidity.

**Calculation:** `df_features['Current Ratio'] = df_features['Total Current Assets'] / df_features['Total Current Liabilities']`

**Division by zero check:** If `Total Current Liabilities` is zero, the ratio will be `inf` or `NaN`. We will keep these values as they are, representing extremely high liquidity or an undefined state, which can be handled during modeling if needed.

In [9]:
# Calculate Current Ratio
# Feature: Current Ratio = Total Current Assets / Total Current Liabilities
# Check for division by zero: If 'Total Current Liabilities' is 0, the ratio will be infinity or NaN.
# We will let pandas handle this naturally (inf for non-zero numerator, NaN for 0/0).
df_features['Current Ratio'] = df_features['Total Current Assets'] / df_features['Total Current Liabilities']

display(df_features[['Ticker', 'Fiscal Year', 'Total Current Assets', 'Total Current Liabilities', 'Current Ratio']].head())

,Ticker,Fiscal Year,Total Current Assets,Total Current Liabilities,Current Ratio
0,ABBV,2020,2.417300e+10,2.866100e+10,0.843411
1,ABBV,2021,2.792800e+10,3.519400e+10,0.793544
2,ABBV,2022,2.846300e+10,2.953800e+10,0.963606
3,ABBV,2023,3.300200e+10,3.784100e+10,0.872123
4,ABBV,2024,2.558200e+10,3.874900e+10,0.660198


In [13]:
df_features.shape

(500, 14)

### 2. Debt-to-Equity Ratio

**Meaning:** Total Liabilities / Total Equity

**Documentation:** This ratio indicates the proportion of equity and debt used to finance a company's assets. A higher ratio generally means more reliance on debt.

**Calculation:** `df_features['Debt-to-Equity'] = df_features['Total Liabilities'] / df_features['Total Equity']`

**Division by zero check:** If `Total Equity` is zero, the ratio will be `inf` or `NaN`. These cases are important signals about a company's financial structure.

In [10]:
# Calculate Debt-to-Equity Ratio
# Feature: Debt-to-Equity = Total Liabilities / Total Equity
# Check for division by zero: If 'Total Equity' is 0, the ratio will be infinity or NaN.
df_features['Debt-to-Equity'] = df_features['Total Liabilities'] / df_features['Total Equity']

display(df_features[['Ticker', 'Fiscal Year', 'Total Liabilities', 'Total Equity', 'Debt-to-Equity']].head())

,Ticker,Fiscal Year,Total Liabilities,Total Equity,Debt-to-Equity
0,ABBV,2020,1.374680e+11,13097000000,10.496144
1,ABBV,2021,1.310930e+11,15436000000,8.492679
2,ABBV,2022,1.215180e+11,17287000000,7.029444
3,ABBV,2023,1.243140e+11,10397000000,11.956718
4,ABBV,2024,1.317970e+11,3364000000,39.178656


In [11]:
df_features.shape

(500, 15)

### 3. Asset Growth

**Meaning:** Percentage change in total assets from the previous fiscal year.

**Documentation:** This measures the rate at which a company's total assets are increasing or decreasing, indicating expansion or contraction.

**Calculation:** `df_features.groupby('Ticker')['Total Assets'].pct_change()`

**Important:** Grouping by `Ticker` and using `pct_change()` ensures that the growth is calculated correctly for each company year-over-year, respecting the "no future information" constraint as `pct_change()` uses the prior period's value.

In [12]:
# Calculate Asset Growth
# Feature: Asset Growth = (Total Assets_current_year - Total Assets_previous_year) / Total Assets_previous_year
# Calculated using `pct_change()` grouped by Ticker to ensure year-over-year calculation.
df_features['Asset Growth'] = df_features.groupby('Ticker')['Total Assets'].pct_change()

display(df_features[['Ticker', 'Fiscal Year', 'Total Assets', 'Asset Growth']].head())

,Ticker,Fiscal Year,Total Assets,Asset Growth
0,ABBV,2020,150565000000,NaN
1,ABBV,2021,146529000000,-0.026806
2,ABBV,2022,138805000000,-0.052713
3,ABBV,2023,134711000000,-0.029495
4,ABBV,2024,135161000000,0.003340


### 4. Equity Growth

**Meaning:** Percentage change in shareholder equity from the previous fiscal year.

**Documentation:** This measures the rate of increase or decrease in the owners' stake in the company.

**Calculation:** `df_features.groupby('Ticker')['Total Equity'].pct_change()`

**Important:** Similar to Asset Growth, this is calculated year-over-year for each company.

In [13]:
# Calculate Equity Growth
# Feature: Equity Growth = (Total Equity_current_year - Total Equity_previous_year) / Total Equity_previous_year
# Calculated using `pct_change()` grouped by Ticker.
df_features['Equity Growth'] = df_features.groupby('Ticker')['Total Equity'].pct_change()

display(df_features[['Ticker', 'Fiscal Year', 'Total Equity', 'Equity Growth']].head())

,Ticker,Fiscal Year,Total Equity,Equity Growth
0,ABBV,2020,13097000000,NaN
1,ABBV,2021,15436000000,0.178591
2,ABBV,2022,17287000000,0.119914
3,ABBV,2023,10397000000,-0.398565
4,ABBV,2024,3364000000,-0.676445


### 5. Debt Growth

**Meaning:** Percentage change in total debt (Total Liabilities) from the previous fiscal year.

**Documentation:** This measures how quickly a company is taking on or paying off debt.

**Calculation:** `df_features.groupby('Ticker')['Total Liabilities'].pct_change()`

**Important:** Calculated year-over-year for each company.

In [14]:
# Calculate Debt Growth
# Feature: Debt Growth = (Total Liabilities_current_year - Total Liabilities_previous_year) / Total Liabilities_previous_year
# Calculated using `pct_change()` grouped by Ticker.
df_features['Debt Growth'] = df_features.groupby('Ticker')['Total Liabilities'].pct_change()

display(df_features[['Ticker', 'Fiscal Year', 'Total Liabilities', 'Debt Growth']].head())

,Ticker,Fiscal Year,Total Liabilities,Debt Growth
0,ABBV,2020,1.374680e+11,NaN
1,ABBV,2021,1.310930e+11,-0.046374
2,ABBV,2022,1.215180e+11,-0.073040
3,ABBV,2023,1.243140e+11,0.023009
4,ABBV,2024,1.317970e+11,0.060194


### Feature Engineering Summary

All requested balance sheet features have been calculated and added to the `df_features` DataFrame. We will now display the first few rows of the `df_features` DataFrame with the newly added features and save it to a new CSV file to avoid overwriting original data.

In [15]:
# Display the DataFrame with all new features
print("DataFrame with new features:")
display(df_features.head(10))




DataFrame with new features:


,Ticker,SimFinId,Fiscal Year,"Cash, Cash Equivalents & Short Term Investments",Total Current Assets,Total Assets,Total Current Liabilities,Total Liabilities,Total Equity,balance_diff,balance_error_pct,Short Term Debt,Long Term Debt,Current Ratio,Debt-to-Equity,Asset Growth,Equity Growth,Debt Growth
0,ABBV,61199,2020,8.479000e+09,2.417300e+10,150565000000,2.866100e+10,1.374680e+11,13097000000,0.0,0.0,8.502000e+09,7.755400e+10,0.843411,10.496144,NaN,NaN,NaN
1,ABBV,61199,2021,9.830000e+09,2.792800e+10,146529000000,3.519400e+10,1.310930e+11,15436000000,0.0,0.0,1.249500e+10,6.418900e+10,0.793544,8.492679,-0.026806,0.178591,-0.046374
2,ABBV,61199,2022,9.229000e+09,2.846300e+10,138805000000,2.953800e+10,1.215180e+11,17287000000,0.0,0.0,4.136000e+09,5.913500e+10,0.963606,7.029444,-0.052713,0.119914,-0.073040
3,ABBV,61199,2023,1.281600e+10,3.300200e+10,134711000000,3.784100e+10,1.243140e+11,10397000000,0.0,0.0,7.191000e+09,5.219400e+10,0.872123,11.956718,-0.029495,-0.398565,0.023009
4,ABBV,61199,2024,5.555000e+09,2.558200e+10,135161000000,3.874900e+10,1.317970e+11,3364000000,0.0,0.0,6.804000e+09,6.034000e+10,0.660198,39.178656,0.003340,-0.676445,0.060194
5,ABNB,1214552,2020,6.391257e+09,8.916386e+09,10491499000,5.139779e+09,7.589716e+09,2901783000,0.0,0.0,NaN,1.815562e+09,1.734780,2.615535,NaN,NaN,NaN
6,ABNB,1214552,2021,8.322000e+09,1.238600e+10,13708000000,6.359000e+09,8.933000e+09,4775000000,0.0,0.0,NaN,1.983000e+09,1.947791,1.870785,0.306582,0.645540,0.176987
7,ABNB,1214552,2022,9.622000e+09,1.486100e+10,16038000000,7.978000e+09,1.047800e+10,5560000000,0.0,0.0,NaN,1.987000e+09,1.862748,1.884532,0.169974,0.164398,0.172954
8,ABNB,1214552,2023,1.007100e+10,1.650900e+10,20645000000,9.950000e+09,1.248000e+10,8165000000,0.0,0.0,NaN,1.991000e+09,1.659196,1.528475,0.287255,0.468525,0.191067
9,ABNB,1214552,2024,1.061100e+10,1.718000e+10,20959000000,1.016100e+10,1.254700e+10,8412000000,0.0,0.0,NaN,1.995000e+09,1.690778,1.491560,0.015209,0.030251,0.005369


In [16]:
# Check for any unexpected missing values in the newly created features
print("\nMissing values in new features:")
display(df_features[['Current Ratio', 'Debt-to-Equity', 'Asset Growth', 'Equity Growth', 'Debt Growth']].isnull().sum())



Missing values in new features:


,0
Current Ratio,0
Debt-to-Equity,0
Asset Growth,100
Equity Growth,100
Debt Growth,100


In [18]:
df_features['Ticker'].isnull().sum()

np.int64(0)

In [25]:
# Save the result as a new CSV file
output_path = 'balance_selected_100_companies_features.csv'
df_features.to_csv(output_path, index=False)
print(f"\nFeatures saved to '{output_path}'")


Features saved to 'balance_selected_100_companies_features.csv'


In [20]:
df_features.shape

(500, 18)

In [19]:
# Identify rows with any missing values in the new features
missing_values_df = df_features[df_features[['Current Ratio', 'Debt-to-Equity', 'Asset Growth', 'Equity Growth', 'Debt Growth']].isnull().any(axis=1)]

# Display the SimFinId, Ticker, Fiscal Year and the new features columns for the companies with missing values
print("Companies with missing values in new features:")
display(missing_values_df[['SimFinId', 'Ticker', 'Fiscal Year', 'Current Ratio', 'Debt-to-Equity', 'Asset Growth', 'Equity Growth', 'Debt Growth']])

Companies with missing values in new features:


,SimFinId,Ticker,Fiscal Year,Current Ratio,Debt-to-Equity,Asset Growth,Equity Growth,Debt Growth
0,61199,ABBV,2020,0.843411,10.496144,NaN,NaN,NaN
5,1214552,ABNB,2020,1.734780,2.615535,NaN,NaN,NaN
10,63877,ABT,2020,1.716721,1.198224,NaN,NaN,NaN
15,919941,AC,2020,1.039214,0.059851,NaN,NaN,NaN
20,14099,ADBE,2020,1.477866,0.830820,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
475,970570,VIRT,2020,1.269182,4.372288,NaN,NaN,NaN
480,120376,WBD,2020,1.988968,1.752726,NaN,NaN,NaN
485,378239,WM,2020,0.996341,2.936812,NaN,NaN,NaN
490,239962,WMT,2020,0.972173,1.884647,NaN,NaN,NaN


In [20]:
missing_values_df.to_csv('missing_values_df.csv', index=False)